<a href="https://colab.research.google.com/github/Amber-0117/Test0207/blob/main/2D_CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install tensorflow

In [ ]:
import os

print(os.getcwd())
print(os.listdir('/content'))

In [ ]:
from numpy import array
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import Flatten

import tensorflow as tf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

tf.random.set_seed(1)

#Data Load
import os

# 指定四季csv
season_files = {
    'autumn': '/content/clearautumn.csv',
    'spring': '/content/clearspring.csv',
    'summer': '/content/clearsummer.csv',
    'winter': '/content/clearwinter.csv'
}

output_dir = '/content/sample_data/season_split'
os.makedirs(output_dir, exist_ok=True)

season_data = {}

for season, file_path in season_files.items():
    df = pd.read_csv(file_path)

    # 預測目標：下一小時的 windspeed_120
    df['target_windspeed_1h'] = df['windspeed_120'].shift(-1)

    # 最後一筆沒有下一小時資料，所以刪掉
    df = df.dropna(subset=['target_windspeed_1h'])

    # 時間資料不打亂，前 80% train，後 20% test
    split_index = int(len(df) * 0.8)

    df_train = df.iloc[:split_index]
    df_test = df.iloc[split_index:]

    # 另存成 CSV
    train_path = f'{output_dir}/{season}_train_80.csv'
    test_path = f'{output_dir}/{season}_test_20.csv'

    df_train.to_csv(train_path, index=False)
    df_test.to_csv(test_path, index=False)

    # 模型輸入：拿掉時間與答案欄位
    train_data_input = df_train.drop(columns=['valid_time', 'target_windspeed_1h'])
    train_data_output = df_train['target_windspeed_1h']

    test_data_input = df_test.drop(columns=['valid_time', 'target_windspeed_1h'])
    test_data_output = df_test['target_windspeed_1h']

    season_data[season] = {
        'df_train': df_train,
        'df_test': df_test,
        'train_data_input': train_data_input,
        'train_data_output': train_data_output,
        'test_data_input': test_data_input,
        'test_data_output': test_data_output
    }

    print(f'===== {season} =====')
    print('train input:', train_data_input.shape)
    print('train output:', train_data_output.shape)
    print('test input:', test_data_input.shape)
    print('test output:', test_data_output.shape)
    print('saved:', train_path)
    print('saved:', test_path)

In [ ]:
# Hyper Parameter
seqLength = 6
filters = 100
kernel_size = (2, 2)
kernel_size2 = (2, 2)
pooling_size = (2, 2)
stride = (1, 1)
epochs = 100
batch_size = 32

target_col = 'target_windspeed_1h'


def sliding_window(X, y, seq_length):
    dataX = []
    dataY = []

    for i in range(0, len(X) - seq_length + 1):
        dataX.append(X[i:i + seq_length])
        dataY.append(y[i + seq_length - 1])

    return np.array(dataX), np.array(dataY).reshape(-1, 1)


def build_2d_cnn(seq_length, feature_count):
    model = Sequential()
    model.add(Conv2D(
        filters=filters,
        kernel_size=kernel_size,
        strides=stride,
        activation='relu',
        input_shape=(seq_length, feature_count, 1)
    ))
    model.add(Conv2D(
        filters=filters,
        kernel_size=kernel_size2,
        strides=stride,
        activation='relu'
    ))
    model.add(MaxPooling2D(pool_size=pooling_size))
    model.add(Flatten())
    model.add(Dense(48, activation='relu'))
    model.add(Dense(1))

    model.compile(optimizer='adam', loss='mse')
    return model


results = []
predictions = {}

plt.figure(figsize=(16, 10))

for idx, season in enumerate(['spring', 'summer', 'autumn', 'winter'], start=1):
    df_train = season_data[season]['df_train']
    df_test = season_data[season]['df_test']

    X_train = df_train.drop(columns=['valid_time', target_col])
    y_train = df_train[target_col]

    X_test = df_test.drop(columns=['valid_time', target_col])
    y_test = df_test[target_col]

    # 全部轉數字
    X_train = X_train.apply(pd.to_numeric, errors='coerce')
    X_test = X_test.apply(pd.to_numeric, errors='coerce')
    y_train = pd.to_numeric(y_train, errors='coerce')
    y_test = pd.to_numeric(y_test, errors='coerce')

    # 補 NaN
    X_train = X_train.fillna(X_train.median()).fillna(0)
    X_test = X_test.fillna(X_train.median()).fillna(0)
    y_train = y_train.fillna(y_train.median())
    y_test = y_test.fillna(y_train.median())

    # MinMax normalization，只用 train 的 min/max
    train_min = X_train.min()
    train_max = X_train.max()
    denom = train_max - train_min
    denom[denom == 0] = 1

    trainSet = ((X_train - train_min) / denom).values
    testSet = ((X_test - train_min) / denom).values

    trainLabel = y_train.values
    testLabel = y_test.values

    trainX, trainY = sliding_window(trainSet, trainLabel, seqLength)
    testX, testY = sliding_window(testSet, testLabel, seqLength)

    trainX = trainX.reshape(trainX.shape[0], trainX.shape[1], trainX.shape[2], 1)
    testX = testX.reshape(testX.shape[0], testX.shape[1], testX.shape[2], 1)

    model = build_2d_cnn(seqLength, trainX.shape[2])

    print(f'===== Training {season} model =====')
    model.fit(
        trainX,
        trainY,
        epochs=epochs,
        batch_size=batch_size,
        verbose=0
    )

    yhat = model.predict(testX, verbose=0).reshape(-1)
    real = testY.reshape(-1)

    mae = mean_absolute_error(real, yhat)
    rmse = np.sqrt(mean_squared_error(real, yhat))
    r2 = r2_score(real, yhat)

    results.append({
        'season': season,
        'MAE': mae,
        'RMSE': rmse,
        'R2': r2
    })

    predictions[season] = {
        'real': real,
        'estimated': yhat,
        'model': model
    }

    plt.subplot(2, 2, idx)
    plt.plot(yhat, label='Estimated by 2D-CNN')
    plt.plot(real, label='Real')
    plt.title(f'{season.capitalize()} One-hour Ahead Wind Speed')
    plt.xlabel('Test sample index')
    plt.ylabel('Wind speed')
    plt.legend()

plt.tight_layout()
plt.show()

results_df = pd.DataFrame(results)
print(results_df)

In [ ]:
train_data_input = season_data['autumn']['train_data_input']
train_data_output = season_data['autumn']['train_data_output']

test_data_input = season_data['autumn']['test_data_input']
test_data_output = season_data['autumn']['test_data_output']

In [ ]:
#Normalization
def MinMaxScaler(data):
  denom = np.max(data,0) - np.min(data,0)
  nume = data-np.min(data,0)
  return nume/denom

def back_MinMax(data,max,min):
  diff = max-min
  back = data * diff + min
  return back



In [ ]:
#Hyper parameters
seqLength = 10
outputDim = 1
hiddenDim = 10
lr = 0.001
iterations = 10
batch_size = 32

target_col = 'target_windspeed_1h'

# X = 輸入資料，不要 valid_time，也不要答案欄位
X_train = df_train.drop(columns=['valid_time', target_col])
y_train = df_train[target_col]

X_test = df_test.drop(columns=['valid_time', target_col])
y_test = df_test[target_col]

# 確保全部都是數字
X_train = X_train.apply(pd.to_numeric, errors='coerce')
X_test = X_test.apply(pd.to_numeric, errors='coerce')
y_train = pd.to_numeric(y_train, errors='coerce')
y_test = pd.to_numeric(y_test, errors='coerce')

# 如果有 NaN，先補掉
X_train = X_train.fillna(X_train.median())
X_test = X_test.fillna(X_train.median())
y_train = y_train.fillna(y_train.median())
y_test = y_test.fillna(y_train.median())

# 用 train 的最大最小值做正規化，test 也用 train 的標準
train_min = X_train.min()
train_max = X_train.max()
denom = train_max - train_min
denom[denom == 0] = 1

trainSet = (X_train - train_min) / denom
testSet = (X_test - train_min) / denom

# 轉 numpy
trainSet = trainSet.values
testSet = testSet.values
trainLabel = y_train.values
testLabel = y_test.values

dataDim = trainSet.shape[1]

print("trainSet:", trainSet.shape)
print("testSet:", testSet.shape)
print("trainLabel:", trainLabel.shape)
print("testLabel:", testLabel.shape)
print("dataDim:", dataDim)

In [ ]:
#Data Windowing
def sliding_window(X, y, seq_length):
    dataX = []
    dataY = []

    for i in range(0, len(X) - seq_length + 1):
        _x = X[i:i + seq_length]
        _y = y[i + seq_length - 1]

        dataX.append(_x)
        dataY.append(_y)

    return np.array(dataX), np.array(dataY).reshape(-1, 1)


trainX, trainY = sliding_window(trainSet, trainLabel, seqLength)
testX, testY = sliding_window(testSet, testLabel, seqLength)

print("trainX:", trainX.shape)
print("trainY:", trainY.shape)
print("testX:", testX.shape)
print("testY:", testY.shape)

In [ ]:
trainX = trainX.reshape(trainX.shape[0], trainX.shape[1], trainX.shape[2], 1)
testX = testX.reshape(testX.shape[0], testX.shape[1], testX.shape[2], 1)

print("CNN trainX:", trainX.shape)
print("CNN testX:", testX.shape)

In [ ]:
from tensorflow.keras.layers import Dense, Flatten, Conv2D, MaxPooling2D

In [ ]:
# Hyper Parameter
filters = 100
kernel_size = (2, 2)
pooling_size = (2, 2)
kernel_size2 = (2, 2)
epochs = 100
stride = (1, 1)

input_shape = trainX.shape[2]

# 2D-CNN Model
model = Sequential()
model.add(Conv2D(
    filters=filters,
    kernel_size=kernel_size,
    strides=stride,
    activation='relu',
    input_shape=(seqLength, input_shape, 1)
))
model.add(Conv2D(
    filters=filters,
    kernel_size=kernel_size2,
    strides=stride,
    activation='relu'
))
model.add(MaxPooling2D(pool_size=pooling_size))
model.add(Flatten())
model.add(Dense(48, activation='relu'))
model.add(Dense(1))

model.compile(optimizer='adam', loss='mse')
model.summary()

In [ ]:
model.fit(trainX,trainY, epochs = epochs, verbose = 0)

In [ ]:
# Evaluate Testset
res2 = model.evaluate(testX, testY, batch_size=batch_size)
print("Test MSE:", res2)

# Predict
yhat = model.predict(testX)

estimated_testY = yhat.reshape(-1)
real = testY.reshape(-1)

print("real.shape:", real.shape)
print("estimated_testY.shape:", estimated_testY.shape)

plt.figure(figsize=(12, 5))
plt.plot(estimated_testY, label="Estimated by 2D-CNN")
plt.plot(real, label="Real")
plt.legend(prop={'size': 10})
plt.show()

print("1132")

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Predict
yhat = model.predict(testX)

estimated_testY = yhat.reshape(-1)
real = testY.reshape(-1)

# Metrics
mae = mean_absolute_error(real, estimated_testY)
mse = mean_squared_error(real, estimated_testY)
rmse = np.sqrt(mse)
r2 = r2_score(real, estimated_testY)

print("MAE:", mae)
print("RMSE:", rmse)
print("R^2:", r2)